## Example 2: Electromagnetic GPR Model Generation

This example demonstrates the use of **ModGen2D** to generate spatially correlated electromagnetic property fields for a typical **ground-penetrating radar (GPR)** modeling scenario. Refer **detailed version** for a step-wise description.

### Model Replication and Automation
This **multiple version** of the example demonstrates how to generate multiple model realizations in a systematic and reproducible manner.

Multiple realizations can be generated efficiently by repeatedly calling the function with different model_id values by defining a random number generator (RNG) seed.

### Notes on Random Number Generator (RNG) Usage
There are two common approaches for using RNGs to generate multiple realizations:

1. Single RNG instance (used in Example 1)
2. Independent RNG per realization, where rng = f(model_id) (recommended; and used here.)
3. 
For this and other examples, the individual cells presented in the detailed version can be consolidated into a single model-generation function. In this approach, a separate random number generator (RNG) can be assigned to each model realization (Method 2).

Overall, this approach enhances reproducibility, scalability, and code clarity, while preserving the same modeling logic presented in the detailed version.

In [1]:
import pandas as pd
import numpy as np
import modgen2d as mg2d
import example2_helper_functions as hf

In [2]:
def generate_one_model(model_id):
    print(f"Generating {model_id}")

    ### ------------------------------------------------------------
    ### Step 1: Length Configuration
    length_config = mg2d.LengthConfig("m", max_grid_density=1000) 
    
    ### ------------------------------------------------------------
    ### Step 2: Properties Definition
    ## Step2.1: General Definitions
    
    # Domain dimensions
    x_span = 5          # Domain length in X-direction (domain units)
    z_span = 3          # Domain depth in Z-direction (domain units)
    
    # Grid spacing
    del_xz_interface = 0.2       # Base spacing for the interfaces (Note: for Unifrom, too small dx might make interface flat)
    del_xz_spatial = 0.1    # Base spacing for the domain for soil domain (spatial correlation): spatial correlation is a computationally heavy.
    del_xz_final = 0.01       # Base spacing for the domain for obstacles and final generated model.
    del_xz_obs = del_xz_final / 10  # Finer spacing for obstacles (recommended: 1/10)
    
    # Random number generator (for reproducibility): Use Model_id for different seed.
    rng = np.random.default_rng(seed=model_id)
    
    # Interface interpolation method
    remesh_interp_method = 'linear'
    
    # Spatial correlation parameters
    spatial_theta_x = 100  # Correlation length in X
    spatial_theta_z = 0.5  # Correlation length in Z
    
    ## Step 2.2: Features Configuration
    # Initialize feature configuration
    feature_config_instance =  mg2d.FeaturesConfig()
    
    # Define material distributions for this simple example
    # For this example, there are only one material type for soil - "generic soil", and utilities - "PVC"
    # Note: these distributions must be random_generators: Hence, using "Constant".
    soil_materials_distribution = mg2d.random_generators.Constant(val = 'sand', rng=rng)
    utils_materials_distribution = mg2d.random_generators.Constant(val = 'PVC', rng=rng)
    
    # Add features to feature_config_instance
    feature_config_instance.add_feature('def', soil_materials_distribution, feature_description = 'def means soil.')
    feature_config_instance.add_feature('U', utils_materials_distribution, feature_description = 'utility features')

    
    ## Step 2.3: Main Properties
    #2.3.1 Main Properties config definition
    main_properties_config_instance =  mg2d.MainPropertiesConfig(feature_config_instance, layer0_flag=False)
    
    #2.3.2 Define each MainProperty instance
    main_property_name = 'ec'
    property_desc = 'electric conductivity'
    main_property_instance = mg2d.MainProperty(main_property_name, feature_config_instance, layer0_flag=False, description=property_desc)
    
    #2.3.3  Define wet and dry properties for each features' each materials (including layer 0  for 'def' if flag is True)
    ## For Feature 'def'; material type 'sand'
    corr_coeff = 0.78
    
    # NOTE: This generator produces two values (Ec and dc) in a single go. Hence, additional post-processing is needed to adjust to format later.
    wet_mean_distribution = hf.CorrelatedUniformBivariateXLogY(20, 30, 0.1, 1, corr_coeff, rng)  
    dry_mean_distribution = hf.CorrelatedUniformBivariateXLogY(3, 5, 0.01, 0.01, corr_coeff, rng)
    cov_distribution = mg2d.random_generators.Constant(0.05, rng)
    cov_type = 'cov'
    wet_prop = mg2d.PropertyDistribution(main_property_name,  wet_mean_distribution, cov_distribution, stdev_type=cov_type)
    dry_prop = mg2d.PropertyDistribution(main_property_name,  dry_mean_distribution, cov_distribution, stdev_type=cov_type)
    
    main_property_instance.add_material_property_of_feature(feature_id='def', material_name='sand', #Must match feature_id and material name (as defined in features_config)
                                                            property_distribution_instance=wet_prop,  
                                                            property_distribution_instance_if_dry=dry_prop 
                                                           )
    
    ## For Feature 'utils'; material type 'PVC'
    corr_coeff = 0.78
    wet_mean_distribution = hf.CorrelatedUniformBivariateXLogY(3, 5, 0.001, 0.001, corr_coeff, rng)
    dry_mean_distribution = None
    cov_distribution = None
    cov_type = 'cov'
    wet_prop = mg2d.PropertyDistribution(main_property_name,  wet_mean_distribution, cov_distribution, stdev_type=cov_type)
    dry_prop = None
    
    main_property_instance.add_material_property_of_feature(feature_id='U', material_name='PVC',
                                                            property_distribution_instance=wet_prop, 
                                                            property_distribution_instance_if_dry=dry_prop)
    
    
    # 2.3.4 Add MainProperty to MainPropertiesConfig instance
    main_properties_config_instance.add_main_property(main_property_instance)
    
    
    
    
    ### Repeat for 'dc'
    ## But since 'dc' is already generated from the bivariate random generator above, so this is defined for a placeholder in sampled properties only.
    #2.3.2 Define each MainProperty instance
    main_property_name = 'dc'
    property_desc = 'dielectric constant'
    main_property_instance = mg2d.MainProperty(main_property_name, feature_config_instance, layer0_flag=False, description=property_desc)
    
    #2.3.3  Define wet and dry properties for each features' each materials (including layer 0  for 'def' if flag is True)
    ## For Feature 'def'; material type 'sand'
    
    # NOTE: Just a placeholder. Additional post-processing is needed to adjust to the format later.
    wet_mean_distribution = mg2d.random_generators.Constant(-999, rng)  # Filler to be replaced later
    dry_mean_distribution = mg2d.random_generators.Constant(-999, rng)  # Filler to be replaced later
    cov_distribution = mg2d.random_generators.Constant(0.05, rng)
    cov_type = 'cov'
    wet_prop = mg2d.PropertyDistribution(main_property_name,  wet_mean_distribution, cov_distribution, stdev_type=cov_type)
    dry_prop = mg2d.PropertyDistribution(main_property_name,  dry_mean_distribution, cov_distribution, stdev_type=cov_type)
    main_property_instance.add_material_property_of_feature(feature_id='def', material_name='sand',
                                                            property_distribution_instance=wet_prop,
                                                            property_distribution_instance_if_dry=dry_prop)
    
    ## For Feature 'utils'; material type 'PVC'
    wet_mean_distribution = mg2d.random_generators.Constant(-999, rng)  # Filler to be replaced later
    dry_mean_distribution = None
    cov_distribution = None
    cov_type = 'cov'
    wet_prop = mg2d.PropertyDistribution(main_property_name,  wet_mean_distribution, cov_distribution, stdev_type=cov_type)
    dry_prop = None
    
    main_property_instance.add_material_property_of_feature(feature_id='U', material_name='PVC',
                                                            property_distribution_instance=wet_prop, 
                                                            property_distribution_instance_if_dry=dry_prop)
    
    # 2.3.4 Add MainProperty to MainPropertiesConfig instance
    main_properties_config_instance.add_main_property(main_property_instance)
    
    # main_properties_config_instance.print()

    ## Step 2.4: Auxiliary Properties
    # Define some additional Properties
    aux_props = mg2d.AuxillaryProperties()
    aux_props.add_aux_property('n_layers',  mg2d.random_generators.DiscreteChoice(x=[4], p=[1], rng=rng))
    aux_props.add_aux_property('gwt', mg2d.random_generators.Uniform(2.4, 2.4, rng))
    
    utilities_sett={
        'n_obs': mg2d.random_generators.DiscreteChoice([0,1], p = [0.2, 0.8], rng=rng),
        'obs_shape': mg2d.random_generators.DiscreteChoice(['circ2d', 'rect2d',], [1/2, 1/2], rng=rng),
        'dia_obs':mg2d.random_generators.Uniform(0.3, 1, rng=rng), 
        'lh_obs':mg2d.random_generators.Uniform(0.3, 1, rng=rng), 
        'obs_x_coord':mg2d.random_generators.Uniform(0, x_span, rng=rng), 
        'depth_top_dist':hf.Discrete2ContinuousPDF([0,5], p = [1,.2], new_del_x = 1, rng=rng), #Continuous distribution: discrete to be converted into continuous: also
    }
    
    aux_props.add_aux_property('n_obs', utilities_sett['n_obs'])
    aux_props.add_aux_property('obs_shape', utilities_sett['obs_shape'])
    aux_props.add_aux_property('dia_obs', utilities_sett['dia_obs'])
    aux_props.add_aux_property('lh_obs', utilities_sett['lh_obs'])
    aux_props.add_aux_property('obs_x_coord', utilities_sett['obs_x_coord'])
    aux_props.add_aux_property('depth_top_utils', utilities_sett['depth_top_dist'])
        
    # aux_props.print()

    ### ------------------------------------------------------------
    ### Step 3: Model Definition
    ## Step 3.1: 2D Domain Definition
    domain_interface = mg2d.DiscretizedDomain2D(x_span, z_span, del_xz_interface, del_xz_interface, length_config)
    domain_spatial = mg2d.DiscretizedDomain2D(x_span, z_span, del_xz_spatial, del_xz_spatial, length_config)
    domain_final = mg2d.DiscretizedDomain2D(x_span, z_span, del_xz_final, del_xz_final, length_config)
    
    ## Step 3.2: Interface Definitions
    # If number of layers > length of list, last value is reused
    roughness_multipliers = [0,1.3,1.2,1]
    
    interface_sett= {
        'generate_surface':False,  # Generate ground surface
    
        # Parameters for step 1: Generation of rough interfaces
        'rough_interface_generator_instance':mg2d.interface.rough_interface_generator.UniformInterfaceGen(1, generate_surface=False, roughness_multipliers=roughness_multipliers),
    
        # Parameters for step 2: Filtering
        'savgol2d_smoother_settings': {
                     'filter_window_length':21, # must be odd
                     'filter_polyorder':7,
                            },
    
        # Parameter for Step 3: Interface Initial Points Generation
        'interfaces_depths_updater':[0, 0.4, 1, 2.3],  # Can be 'random', 'equidistant', or np.ndarray (skips zs generation.). Default: 'random',
        'interfaces_depth_reference_point_x':2.5,  # Default: None,
        
        # Parameter for Step 4: Handling the overlapping and adjust surface_top_to_zero
        'overlapping_resolver_technique': 'erosion', # Options: 'erosion', 'reverse_erosion'. Default: 'erosion'
        'adjust_surface_top_to_zero': False,  # Default: True
        }

    ## Step 3.2: Interface Definitions (Continued)
    n_layers = aux_props.aux_properties['n_layers'].generate()
    gwt_depth = aux_props.aux_properties['gwt'].generate()
    
    # DiscretizedInterfaces2D from dictionary definition
    soil_interface = mg2d.interface.DiscretizedInterfaces2DFromDict(domain_interface, n_layers, interface_sett, remesh_interp_method=remesh_interp_method, rng=rng)
    # soil_interface.plot()

    ### Step 3.3: Lithological Domain (2D) Definition
    ## Step 3.3.1: From Interfaces (Global Soil Interface Configuration)
    
    # Reset global soil interface configuration (safety step)
    mg2d.GlobalSoilInterfaceConfig.reset()   # For safety only
    
    # Register soil interface
    mg2d.GlobalSoilInterfaceConfig.set_soil_interface(soil_interface)
    
    ## Get lithological domain from interface
    name = 'soil_lit'
    lit = mg2d.LithologicalDomain2D(domain_spatial, gwt_depth, name)
    # lit.plot(discrete_point_size = 10, plot_interfaces=True)
    
    ## Step 3.3.2: From Obstruction2D (Global Soil Interface Configuration)
    ## Define a LithologicalDoamin2d From obstruction 2d.
    obs_lit = mg2d.LithologicalDomain2DFromObstruction2D(domain_final, 'obstructions')
    
    # Number of obstructions to generate
    n_obs = aux_props.aux_properties['n_obs'].generate()
    
    # Randomly generate obstruction shapes
    obs_shapes = aux_props.aux_properties['obs_shape'].generate((n_obs,))
    
    # For each obstruction, create a Obstruction2D instance first, and then add to obs_lit. 
    for i, obs_shape in enumerate(obs_shapes):
        # Generate obstruction location
        obs_x_coord = aux_props.aux_properties['obs_x_coord'].generate()
        d_obs = aux_props.aux_properties['depth_top_utils'].generate()
    
        # Unique obstruction ID
        obs_id = i+1
    
        # Initialize Obstruction2D object
        obs_instance = mg2d.Obstruction2D(dl = del_xz_obs, ref_xz_symbolic = ['c', 0], snap_to_dl=True)
    
        # Define obstruction geometry
        if obs_shape == 'circ2d':
            d = aux_props.aux_properties['dia_obs'].generate()
            obs_instance.circle_2d(d, obstruction_id = obs_id)
        elif obs_shape == 'rect2d':
            lx = aux_props.aux_properties['lh_obs'].generate()
            lz = aux_props.aux_properties['lh_obs'].generate()
            obs_instance.rectangle_2d(lx, lz, obstruction_id = obs_id)
        else:
            raise ValueError(f"Invalid util_shape {obs_shape}")
        # obs_instance.plot()
    
        ## Add Obs2D into defnined lit_domain_from_obs2d 
        obs_lit.add_obstruction2D(obs_instance, shift_ref2d_to_xy = [obs_x_coord, d_obs], feature_id='U')
    
        # Plot lithological domain generated from obstructions (JUST FOR VISUALIZATION)
        # obs_lit.plot()
        
    
    # Visualize merged lithological domain (JUST FOR VISUALIZATION)
    merged_lit = lit.return_merged_lithological_domain([obs_lit], warn_if_null_lithological_domain=False)  # Just to see how merged lit. Karst to be added later.
    # merged_lit.plot(discrete_point_size = 10, plot_interfaces=True)
    
    ## Step 3.4: Lithological Domain Collection
    
    # Initialize lithological domain collection
    lit_collection = mg2d.LithologicalDomain2DCollection(main_properties_config_instance.get_feature_ids(), interface_set_name="soil") 
    
    # Add soil-based lithological domain
    lit_collection.add_lithological_domain_from_soil_interface_config(lit)
    lit_collection.add_lithological_domain_from_obstruction2d("obs", obs_lit, warn_if_null_lithological_domain=False)
    
    # Finalize and lock the lithological domain collection
    lit_collection.lock()

    ### ------------------------------------------------------------
    ### Step 4: Generate Simulated Property Profiles
    # Unlock main properties config to allow sampling
    main_properties_config_instance.unlock()
    
    # Sample property values for all features in the lithological domain
    main_properties_config_instance.lock_and_generate_sample_properties(lit_collection)
    sample_properties = main_properties_config_instance.sampled_properties

    #As illustrated above, two values are generated for 'ec', while 'dc' currently contains only a placeholder. 
    #Nevertheless, the data format remains consistent. In other words, soil properties are represented as 'wet' and 'dry',
    # whereas utilities use 'both'. Maintaining this consistent structure is essential. 
    # The following steps perform post-processing to adjust and standardize the data format.

    for lit_id in sample_properties['ec'].keys():
        for case in sample_properties['ec'][lit_id].keys():
            ec_dc = sample_properties['ec'][lit_id][case]['mean']
            ec, dc = float(ec_dc[0][0]), float(ec_dc[0][1])
            sample_properties['ec'][lit_id][case]['mean'] = ec
            sample_properties['dc'][lit_id][case]['mean'] = dc
    main_properties_config_instance._sampled_properties = sample_properties
    
    # Initialize spatial simulator (controls spatial correlation)
    spatial_sim = mg2d.spatial_simulator2d.CovarianceDecompositionSimulator(spatial_theta_x, spatial_theta_z, rng = rng)
    
    # Generate property profiles using the spatial simulator
    gen_profiles = mg2d.GeneratedProfileCollection2D(main_properties_config_instance, lit_collection, spatial_sim)
    gen_profiles.simulate_property_profile('dc')        
    gen_profiles.simulate_property_profile('ec')        
    
    #Save the profiles into h5 files
    i = model_id
    gen_profiles.save_to_hdf5(f'generated_h5_files/{i:08d}.h5', hdf5_compression_level=8)

In [3]:
for model_id in range(1,25):
    generate_one_model(model_id)
    

Generating 1


F:\V_Tech Semesters\MS Thesis\Jupyter_MS\git\modgen2d\modgen2d\interface\_read_only.py:101: UserWarning: The top of surface_interface is not adjusted to zero. RECOMMENDED to use .adjust_top_of_surface_interface_to_zero().
  warnings.warn("The top of surface_interface is not adjusted to zero. RECOMMENDED to use .adjust_top_of_surface_interface_to_zero().")


Z-vals: spatial-with-sigma
Simulating z-vals for Layer ID: 1
Z-vals: spatial-with-sigma
Simulating z-vals for Layer ID: 2
Z-vals: spatial-with-sigma
Simulating z-vals for Layer ID: 3
Z-vals: spatial-with-sigma
Simulating z-vals for Layer ID: 4
Z-vals: spatial-with-sigma
Simulating z-vals for Layer ID: U_1
Simulating z-vals for Layer ID: X


F:\V_Tech Semesters\MS Thesis\Jupyter_MS\git\modgen2d\modgen2d\spatial_simulator2d.py:348: UserWarning: Layer 1 (wet): sigma=0.013100807809297683 but all simulated values are zero.
  warnings.warn(
F:\V_Tech Semesters\MS Thesis\Jupyter_MS\git\modgen2d\modgen2d\spatial_simulator2d.py:348: UserWarning: Layer 3 (wet): sigma=0.010472878054733415 but all simulated values are zero.
  warnings.warn(
F:\V_Tech Semesters\MS Thesis\Jupyter_MS\git\modgen2d\modgen2d\spatial_simulator2d.py:348: UserWarning: Layer 2 (wet): sigma=0.04392034456394964 but all simulated values are zero.
  warnings.warn(
F:\V_Tech Semesters\MS Thesis\Jupyter_MS\git\modgen2d\modgen2d\spatial_simulator2d.py:306: UserWarning: Layer U_1: sigma=0 but 999 z-values are non-zero.
  warnings.warn(


Z-vals: spatial-with-sigma
Simulating z-vals for Layer ID: 1
Z-vals: spatial-with-sigma
Simulating z-vals for Layer ID: 2
Z-vals: spatial-with-sigma
Simulating z-vals for Layer ID: 3
Z-vals: spatial-with-sigma
Simulating z-vals for Layer ID: 4
Z-vals: spatial-with-sigma
Simulating z-vals for Layer ID: U_1
Simulating z-vals for Layer ID: X


...

Z-vals: spatial-with-sigma
Simulating z-vals for Layer ID: 1
Z-vals: spatial-with-sigma
Simulating z-vals for Layer ID: 2
Z-vals: spatial-with-sigma
Simulating z-vals for Layer ID: 3
Z-vals: spatial-with-sigma
Simulating z-vals for Layer ID: 4
Z-vals: spatial-with-sigma
Simulating z-vals for Layer ID: U_1
Simulating z-vals for Layer ID: X


F:\V_Tech Semesters\MS Thesis\Jupyter_MS\git\modgen2d\modgen2d\spatial_simulator2d.py:348: UserWarning: Layer 1 (wet): sigma=0.007700203726208306 but all simulated values are zero.
  warnings.warn(
F:\V_Tech Semesters\MS Thesis\Jupyter_MS\git\modgen2d\modgen2d\spatial_simulator2d.py:348: UserWarning: Layer 2 (wet): sigma=0.028405610124768818 but all simulated values are zero.
  warnings.warn(
F:\V_Tech Semesters\MS Thesis\Jupyter_MS\git\modgen2d\modgen2d\spatial_simulator2d.py:306: UserWarning: Layer U_1: sigma=0 but 3024 z-values are non-zero.
  warnings.warn(


Z-vals: spatial-with-sigma
Simulating z-vals for Layer ID: 1
Z-vals: spatial-with-sigma
Simulating z-vals for Layer ID: 2
Z-vals: spatial-with-sigma
Simulating z-vals for Layer ID: 3
Z-vals: spatial-with-sigma
Simulating z-vals for Layer ID: 4
Z-vals: spatial-with-sigma
Simulating z-vals for Layer ID: U_1
Simulating z-vals for Layer ID: X


F:\V_Tech Semesters\MS Thesis\Jupyter_MS\git\modgen2d\modgen2d\spatial_simulator2d.py:348: UserWarning: Layer 1 (wet): sigma=1.1185520119297814 but all simulated values are zero.
  warnings.warn(
F:\V_Tech Semesters\MS Thesis\Jupyter_MS\git\modgen2d\modgen2d\spatial_simulator2d.py:348: UserWarning: Layer 2 (wet): sigma=1.3102498199076167 but all simulated values are zero.
  warnings.warn(
F:\V_Tech Semesters\MS Thesis\Jupyter_MS\git\modgen2d\modgen2d\spatial_simulator2d.py:306: UserWarning: Layer U_1: sigma=0 but 3024 z-values are non-zero.
  warnings.warn(


Data saved to generated_h5_files/00000024.h5
